# 00 · Colab bootstrap — Plant Disease Detection (Yarpaq Həkimləri)

Run this notebook **first in every new Colab session**. It:

1. checks the GPU,
2. mounts Google Drive so models and logs survive a session reset,
3. clones (or updates) the repo and installs the `plantdx` package,
4. prints an environment report and saves it to Drive,
5. runs a quick check that the PlantVillage index files are reachable and consistent.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

## 1 · GPU

In [ ]:
!nvidia-smi -L || echo "No GPU attached: switch the runtime to T4 GPU before training." 

## 2 · Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3 · Clone or update the repo

In [ ]:
import os

REPO_URL = "https://github.com/AnarHummatli/plant-disease-detection.git"
REPO_DIR = "/content/plant-disease-detection"
BRANCH = "main"

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} fetch -q origin && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q --ff-only
else:
    !git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log -1 --oneline

## 4 · Install `plantdx`

Colab already ships a GPU-ready TensorFlow, so only the core and dev dependencies are installed (no `[tf]` extra).

In [ ]:
%pip install -q -e ".[dev]"

import sys

# An editable install is not visible to an already running kernel; add src/ directly.
if f"{REPO_DIR}/src" not in sys.path:
    sys.path.insert(0, f"{REPO_DIR}/src")

import plantdx

print("plantdx", plantdx.__version__, "from", plantdx.__file__)

## 5 · Artifacts on Drive + environment report

In [ ]:
import json
from datetime import datetime

os.environ["PLANTDX_ARTIFACTS"] = "/content/drive/MyDrive/plantdx/artifacts"

from plantdx.env import report
from plantdx.paths import artifacts_dir

info = report()
for key, value in info.items():
    print(f"{key:>16}: {value}")

env_dir = artifacts_dir() / "env"
env_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
(env_dir / f"colab-{stamp}.json").write_text(json.dumps(info, indent=2, default=str))
print("\nSaved to", env_dir / f"colab-{stamp}.json")

## 6 · Data access check (index files only, ~7 MB)

Downloads the official split lists and the leaf map (not the 2.2 GB image archive), then:

* **official release** (information): sizes against the dataset card, all 38 classes, how many images have a known
  physical-leaf id, and how many leaf ids the official train and test share (227 expected: the reason the splits
  are rebuilt, `docs/decisions.md` D-004);
* **committed splits** (`splits/*.csv`): the test split equals the official one, and no leaf, duplicate group or
  near-identical pair (D-006) is shared between train, val and test.

In [ ]:
from collections import Counter

import pandas as pd

from plantdx.data.source import (
    CLASSES,
    EXPECTED_SIZES,
    fetch_index_files,
    leaf_identifier,
    load_leaf_map,
    read_split_index,
    relative_path,
    resolve_leaf_id,
)
from plantdx.data.split import assert_no_leakage

files = fetch_index_files("color")
train = read_split_index(files["train"])
test = read_split_index(files["test"])
leaf_map = load_leaf_map(files["leaf_map"])


def leaf_ids(records):
    return [resolve_leaf_id(leaf_identifier(r.path), r.label, leaf_map) for r in records]


train_ids, test_ids = leaf_ids(train), leaf_ids(test)
all_ids = train_ids + test_ids
fallback = sum(i.startswith("fallback_") for i in all_ids)
shared = set(train_ids) & set(test_ids)

print("OFFICIAL RELEASE")
print(f"  train images: {len(train):,}   test images: {len(test):,}")
print(f"  images without a known leaf id (fallback): {fallback:,} ({fallback / len(all_ids):.1%})")
print(f"  leaf ids shared by official train and test: {len(shared)} (227 expected, handled by our splits)")
print("  smallest official train classes:", Counter(r.label for r in train).most_common()[-3:])

ours = pd.concat(
    [pd.read_csv(f"splits/{s}.csv").assign(split=s) for s in ("train", "val", "test")], ignore_index=True
)
try:
    assert_no_leakage(ours, pd.read_csv("splits/duplicate_pairs.csv"))
    no_leak = True
except AssertionError as err:
    print(err)
    no_leak = False

checks = {
    "official train size matches the dataset card": len(train) == EXPECTED_SIZES["color"]["train"],
    "official test size matches the dataset card": len(test) == EXPECTED_SIZES["color"]["test"],
    "all 38 classes in the official train and test": {r.label for r in train} == {r.label for r in test} == set(CLASSES),
    "committed test split == official test split": set(ours.loc[ours.split == "test", "path"]) == {relative_path(r.path) for r in test},
    "committed splits: no leaf, group or near-identical pair shared": no_leak,
}
print("\nCOMMITTED SPLITS:", ours["split"].value_counts().to_dict())
for name, ok in checks.items():
    print(("PASS  " if ok else "FAIL  ") + name)

## Done criteria (plan step 1)

- [ ] GPU cell shows a Tesla T4 (or note that this session is CPU-only)
- [ ] `import plantdx` works and the environment report is saved on Drive
- [ ] Data access check: all lines `PASS` (any `FAIL` line → something is wrong; do not train)